# 02 DistilBERT Baselines

This notebook runs the thesis DistilBERT experiments:

- **Baseline 2**: fine-tuned `distilbert-base-cased` trained on the clean training split.
- **Ablation B**: the same DistilBERT setup trained on the adversarially augmented training split.

Both models are evaluated on the clean test set and the 10%, 20%, and 30% adversarial test sets. Outputs are saved under `trained_models/`, `results/metrics/`, `results/predictions/`, `results/figures/`, and `reports/`.

## 1. Paths

Set `BASE_DIR` to the `thesis-modeling` folder. The local default works when this notebook is opened from `thesis-modeling/notebooks/`.

In [ ]:
from pathlib import Path

# Google Drive example:
# BASE_DIR = Path('/content/drive/MyDrive/Thesis_Modeling/thesis-modeling')

# Local fallback: works whether the notebook kernel starts in thesis-modeling/ or thesis-modeling/notebooks/.
cwd = Path.cwd().resolve()
BASE_DIR = cwd.parent if cwd.name == 'notebooks' else cwd
SCRIPT_PATH = BASE_DIR / 'scripts' / 'run_distilbert_baselines.py'
DATA_DIR = BASE_DIR / 'data' / '06_model_ready'

print('BASE_DIR:', BASE_DIR)
print('SCRIPT_PATH exists:', SCRIPT_PATH.exists())
print('DATA_DIR exists:', DATA_DIR.exists())

## 2. Dependency Check

The script uses PyTorch, Hugging Face Transformers, pandas, NumPy, and matplotlib. In Colab, install missing packages before running the full experiment.

In [ ]:
import importlib.util

required = ['torch', 'transformers', 'pandas', 'numpy', 'matplotlib']
missing = [name for name in required if importlib.util.find_spec(name) is None]
print('Missing:', missing)

# Uncomment in Colab if needed:
# %pip install -q torch transformers pandas numpy matplotlib

## 3. Smoke Test

Run this first to verify that loading data, tokenization, training, evaluation, and file writing all work. It uses only a small row subset.

In [ ]:
!python "{SCRIPT_PATH}" --base-dir "{BASE_DIR}" --models baseline2_clean ablation_b_augmented --seeds 42 --epochs 1 --batch-size 16 --smoke-test

## 4. Full Thesis Run

Run this for the actual thesis results. On CPU this can be slow; use a GPU runtime in Colab when possible.

In [ ]:
!python "{SCRIPT_PATH}" --base-dir "{BASE_DIR}" --models baseline2_clean ablation_b_augmented --seeds 42 7 123 --epochs 10 --batch-size 32

## 5. Inspect Results

In [ ]:
import pandas as pd

metrics_path = BASE_DIR / 'results' / 'metrics' / 'distilbert_baselines_full_metrics.csv'
if not metrics_path.exists():
    metrics_path = BASE_DIR / 'results' / 'metrics' / 'distilbert_baselines_smoke_metrics.csv'

metrics = pd.read_csv(metrics_path)
display(metrics)
display(metrics.groupby(['model', 'condition'])[['accuracy', 'precision', 'recall', 'f1', 'fnr', 'fpr']].mean().reset_index())